In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('data'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

data/test.csv
data/train.csv


## Importing essential Libraries

In [ ]:
pd.set_option('display.max_rows', None) 

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
train_ds = pd.read_csv("data/train.csv")
test_ds = pd.read_csv("data/test.csv")

In [ ]:
train_ds.head(5)

In [ ]:
train_ds.info()

In [ ]:
Null_Count = pd.DataFrame(train_ds.isnull().sum())
print(train_ds.shape)

In [ ]:
Null_Count

## Graphs for visualization

In [ ]:
data = pd.concat([train_ds['LotArea'], train_ds['SalePrice']], axis = 1)
data.plot.scatter(x = 'LotArea', y = 'SalePrice')

In [ ]:
data = pd.concat([train_ds['TotalBsmtSF'], train_ds['SalePrice']], axis = 1)
data.plot.scatter(x = 'TotalBsmtSF', y = 'SalePrice')

In [ ]:
data = pd.concat([train_ds['GarageArea'], train_ds['SalePrice']], axis = 1)
data.plot.scatter(x = 'GarageArea', y = 'SalePrice')

In [ ]:
data = pd.concat([train_ds['MasVnrArea'], train_ds['SalePrice']], axis = 1)
data.plot.scatter(x = 'MasVnrArea', y = 'SalePrice')

In [ ]:
data = pd.concat([train_ds['YearBuilt'], train_ds['SalePrice']], axis = 1)
fig = sns.boxplot(x = 'YearBuilt', y = 'SalePrice', data = data)
fig.axis(ymin=0, ymax=800000);

In [ ]:
data = pd.concat([train_ds['OverallQual'], train_ds['SalePrice']], axis = 1)
fig = sns.boxplot(x = 'OverallQual', y = 'SalePrice', data = data)

In [ ]:
#train_ds.hist(figsize=(16, 20), bins=50, xlabelsize=8, ylabelsize=8);

## Data Preprocessing

In [4]:
train_ds.drop(['Id', 'MoSold', 'GarageYrBlt', 'Condition1', 'Condition2'], axis = 1, inplace = True)

In [5]:
for column in train_ds:
    null_count = train_ds[column].isnull().sum()
    if null_count > 1:
        print(f"Dropping column {column} with {null_count} missing values.")
        train_ds.drop(column, axis = 1, inplace = True)

Dropping column LotFrontage with 259 missing values.
Dropping column Alley with 1369 missing values.
Dropping column MasVnrType with 872 missing values.
Dropping column MasVnrArea with 8 missing values.
Dropping column BsmtQual with 37 missing values.
Dropping column BsmtCond with 37 missing values.
Dropping column BsmtExposure with 38 missing values.
Dropping column BsmtFinType1 with 37 missing values.
Dropping column BsmtFinType2 with 38 missing values.
Dropping column FireplaceQu with 690 missing values.
Dropping column GarageType with 81 missing values.
Dropping column GarageFinish with 81 missing values.
Dropping column GarageQual with 81 missing values.
Dropping column GarageCond with 81 missing values.
Dropping column PoolQC with 1453 missing values.
Dropping column Fence with 1179 missing values.
Dropping column MiscFeature with 1406 missing values.


In [6]:
le = LabelEncoder()
string_columns = train_ds.select_dtypes(include = ['object']).columns
for column in string_columns:
    train_ds[column] = le.fit_transform(train_ds[column])

In [ ]:
train_ds.head(5)
#train_ds.shape

## Training

In [7]:
X = train_ds.drop(['SalePrice'], axis = 1)
y = train_ds['SalePrice']

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42)

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

### Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
LR_Model = LinearRegression()

In [ ]:
LR_Model.fit(X_train, y_train)

In [ ]:
y_pred = LR_Model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

In [ ]:
print(f'Mean Absolute Average: {mae}')
print(f'Mean Squared Average: {mse}')
print(f'R2 Score: {r2}')

### Simple Neural Network

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Dense, Input

In [ ]:
model = keras.Sequential()

model.add(Input(shape = (58, )))
model.add(Dense(32, activation = 'relu'))
model.add(Dense(16, activation = 'relu'))
model.add(Dense(1))

model.compile(optimizer = keras.optimizers.Adam(0.015), loss = 'mean_squared_error')

In [ ]:
model.fit(X_train, y_train, epochs = 30, batch_size = 32)

In [ ]:
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f'R2 Score: {r2}')
print(f'MSE: {mse}')

### Decision trees

In [ ]:
from sklearn.tree import DecisionTreeRegressor

reg = DecisionTreeRegressor()

In [ ]:
reg.fit(X_train, y_train)

In [ ]:
y_pred = reg.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f'R2 Score: {r2}')
print(f'MSE: {mse}')

### Random Forest Regressor

In [9]:
from sklearn.ensemble import RandomForestRegressor
FReg = RandomForestRegressor(n_estimators = 100, random_state = 42)

In [10]:
FReg.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [11]:
y_pred = FReg.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f'R2 Score: {r2}')
print(f'MSE: {mse}')

R2 Score: 0.9044780066201905
MSE: 666561426.0361346


### XGBoost

In [ ]:
from xgboost import XGBRegressor
XGB = XGBRegressor(n_estmators = 100, random_state = 42)

In [ ]:
XGB.fit(X_train, y_train)

In [ ]:
y_pred = XGB.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f'R2 Score: {r2}')
print(f'MSE: {mse}')

## Graph for analyzing models

In [ ]:
model_names = ['Linear Regression', 'Neural Network', 'Decision Tree', 'Random Forest', 'XGBoost']  

r2_scores = [0.83985, 0.75706, 0.77352, 0.90447, 0.90048]  


plt.figure(figsize=(10, 6))
plt.bar(model_names, r2_scores, color='red')
plt.xlabel('Models')
plt.ylabel('R-squared (R2) Score')
plt.title('R-squared (R2) Scores for Different Regression Models')
plt.ylim(0, 1) 
plt.xticks(rotation=45)

## Data Preprocessing Test Data


In [12]:
test_ds_ids = test_ds['Id'] # fix for crash isolation purpose
test_ds.drop(['Id', 'MoSold', 'GarageYrBlt', 'Condition1', 'Condition2'], axis = 1, inplace = True)

In [ ]:
Null_Count = pd.DataFrame(train_ds.isnull().sum())
Null_Count

In [13]:
# fix --- test_ds should have the same columns as training dataset, fill nan with 0

# for column in test_ds:
#     null_count = test_ds[column].isnull().sum()
#     if null_count > 1:
#         print(f"Dropping column {column} with {null_count} missing values.")
#         test_ds.drop(column, axis = 1, inplace = True)

test_ds = test_ds[X_train.columns]
for col in test_ds.columns:
    if test_ds[col].dtype == 'object':
        test_ds[col] = test_ds[col].fillna("")
    elif pd.api.types.is_numeric_dtype(test_ds[col]):
        test_ds[col] = test_ds[col].fillna(0)

In [14]:
le = LabelEncoder()
string_columns = test_ds.select_dtypes(include = ['object']).columns
for column in string_columns:
    test_ds[column] = le.fit_transform(test_ds[column])

In [ ]:
test_ds.info()

## RandomForest

In [15]:
predictions = FReg.predict(test_ds)
submissions_df = pd.DataFrame({
    "ID" : test_ds_ids, # test_data['ID'], # fix for crash isolation purpose
    "Predictions" : predictions
})

# submissions_df.to_csv('submission_csv', index = False)